In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

column_names = ['unit', 'cycle', 'op1', 'op2', 'op3'] + [f'sensor{i}' for i in range(1, 22)]
train = pd.read_csv('../data/train_FD001.txt', sep=r'\s+', header=None, names=column_names)

# Sensors identified as constant during EDA (see 01_eda.ipynb)
constant_sensors = ['sensor1', 'sensor5', 'sensor6', 'sensor10', 'sensor16', 'sensor18', 'sensor19']
useful_sensors = [c for c in train.columns if c.startswith('sensor') and c not in constant_sensors]

print(train.shape, len(useful_sensors))

(20631, 26) 14


In [32]:
# Step 1: find each engine's max cycle (its failure point), since train trajectories run to failure.
max_cycle = train.groupby('unit')['cycle'].max().rename('max_cycle')

# Step 2: merge that value back into every row of train, matched by engine (unit) number.
train = train.merge(max_cycle, on='unit')

# Step 3: raw linear RUL = cycles remaining until failure.
train['RUL_raw'] = train['max_cycle'] - train['cycle']

# Step 4: cap RUL at 125 -- degradation signal is weak before this point (see EDA),
# so treating early cycles as a flat "healthy" plateau is more realistic than a linear countdown.
RUL_CAP = 125
train['RUL'] = train['RUL_raw'].clip(upper=RUL_CAP)

train[['unit', 'cycle', 'max_cycle', 'RUL_raw', 'RUL']].head(10)

,unit,cycle,max_cycle,RUL_raw,RUL
0,1,1,192,191,125
1,1,2,192,190,125
2,1,3,192,189,125
3,1,4,192,188,125
4,1,5,192,187,125
5,1,6,192,186,125
6,1,7,192,185,125
7,1,8,192,184,125
8,1,9,192,183,125
9,1,10,192,182,125


In [33]:
engine_1 = train[train['unit'] == 1]
engine_1[['unit', 'cycle', 'RUL_raw', 'RUL']].tail(15)

,unit,cycle,RUL_raw,RUL
177,1,178,14,14
178,1,179,13,13
179,1,180,12,12
180,1,181,11,11
181,1,182,10,10
182,1,183,9,9
183,1,184,8,8
184,1,185,7,7
185,1,186,6,6
186,1,187,5,5


In [40]:
# Split engines (not rows) into train/validation, so no engine's cycles appear in both sets.
# This avoids leakage: predicting a cycle you've already seen nearby cycles for is artificially easy.
np.random.seed(42)

unique_engines = train['unit'].unique()
np.random.shuffle(unique_engines)

n_val_engines = 20
val_engines = unique_engines[:n_val_engines]
train_engines = unique_engines[n_val_engines:]

train_split = train[train['unit'].isin(train_engines)]
val_split = train[train['unit'].isin(val_engines)]

print("Train engines:", train_split['unit'].nunique(), "Train rows:", train_split.shape[0])
print("Validation engines:", val_split['unit'].nunique(), "Validation rows:", val_split.shape[0])

Train engines: 80 Train rows: 16561
Validation engines: 20 Validation rows: 4070


In [41]:
# Verify no engine appears in both train and validation sets.
overlap = set(train_split['unit'].unique()) & set(val_split['unit'].unique())
print("Number of overlapping engines:", len(overlap))

Number of overlapping engines: 0


In [36]:
# Compute a rolling mean per sensor, calculated separately within each engine (groupby),
# so windows never mix data across different engines.
WINDOW = 5

rolling_mean = train.groupby('unit')[useful_sensors].rolling(window=WINDOW, min_periods=1).mean()
rolling_mean = rolling_mean.reset_index(level=0, drop=True)
rolling_mean.columns = [f'{col}_rollmean{WINDOW}' for col in rolling_mean.columns]

train = pd.concat([train, rolling_mean], axis=1)

train[['unit', 'cycle', 'sensor4', 'sensor4_rollmean5']].head(10)

,unit,cycle,sensor4,sensor4_rollmean5
0,1,1,1400.60,1400.600000
1,1,2,1403.14,1401.870000
2,1,3,1404.20,1402.646667
3,1,4,1401.87,1402.452500
4,1,5,1406.22,1403.206000
5,1,6,1398.37,1402.760000
6,1,7,1397.77,1401.686000
7,1,8,1400.97,1401.040000
8,1,9,1394.80,1399.626000
9,1,10,1400.46,1398.474000


In [37]:
# Compute rolling standard deviation per sensor, same window logic as rolling mean.
rolling_std = train.groupby('unit')[useful_sensors].rolling(window=WINDOW, min_periods=1).std()
rolling_std = rolling_std.reset_index(level=0, drop=True)
rolling_std.columns = [f'{col}_rollstd{WINDOW}' for col in rolling_std.columns]
rolling_std = rolling_std.fillna(0)

train = pd.concat([train, rolling_std], axis=1)

train[['unit', 'cycle', 'sensor4', 'sensor4_rollmean5', 'sensor4_rollstd5']].head(10)

,unit,cycle,sensor4,sensor4_rollmean5,sensor4_rollstd5
0,1,1,1400.60,1400.600000,0.000000
1,1,2,1403.14,1401.870000,1.796051
2,1,3,1404.20,1402.646667,1.850009
3,1,4,1401.87,1402.452500,1.559645
4,1,5,1406.22,1403.206000,2.159440
5,1,6,1398.37,1402.760000,2.926337
6,1,7,1397.77,1401.686000,3.648360
7,1,8,1400.97,1401.040000,3.367046
8,1,9,1394.80,1399.626000,4.289514
9,1,10,1400.46,1398.474000,2.458603


In [38]:
print(train.shape)
print([col for col in train.columns if 'rollmean' in col])

(20631, 57)
['sensor2_rollmean5', 'sensor3_rollmean5', 'sensor4_rollmean5', 'sensor7_rollmean5', 'sensor8_rollmean5', 'sensor9_rollmean5', 'sensor11_rollmean5', 'sensor12_rollmean5', 'sensor13_rollmean5', 'sensor14_rollmean5', 'sensor15_rollmean5', 'sensor17_rollmean5', 'sensor20_rollmean5', 'sensor21_rollmean5']


In [42]:
from sklearn.preprocessing import StandardScaler

# Feature columns to normalize: raw sensors + their rolling mean/std versions
rollmean_cols = [f'{s}_rollmean{WINDOW}' for s in useful_sensors]
rollstd_cols = [f'{s}_rollstd{WINDOW}' for s in useful_sensors]
feature_cols = useful_sensors + rollmean_cols + rollstd_cols

scaler = StandardScaler()
scaler.fit(train_split[feature_cols])

train_split_scaled = train_split.copy()
train_split_scaled[feature_cols] = scaler.transform(train_split[feature_cols])

val_split_scaled = val_split.copy()
val_split_scaled[feature_cols] = scaler.transform(val_split[feature_cols])

train_split_scaled[feature_cols].describe().loc[['mean', 'std']]

,sensor2,sensor3,sensor4,sensor7,sensor8,sensor9,sensor11,sensor12,sensor13,sensor14,...,sensor8_rollstd5,sensor9_rollstd5,sensor11_rollstd5,sensor12_rollstd5,sensor13_rollstd5,sensor14_rollstd5,sensor15_rollstd5,sensor17_rollstd5,sensor20_rollstd5,sensor21_rollstd5
mean,-2.197573e-14,-1.421858e-14,-4.400294e-15,-4.518024e-14,-1.117482e-12,1.123757e-14,-3.514743e-15,1.054938e-13,7.276136e-12,3.933492e-15,...,-7.165065e-17,6.864733e-17,1.544565e-16,-8.538011e-17,-9.782244e-17,6.006641e-17,2.059420e-16,1.064034e-16,-8.924152e-17,-5.320168e-17
std,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,...,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00,1.000030e+00
